In [0]:
%run "../Includes/Configuration"

In [0]:
%run "../Includes/Common_Functions"

In [0]:
%run "../Includes/sas alternative"

In [0]:
create database if not exists f_presentation
location "{Presentation_path}";

In [0]:
drop table if exists f_presentation.Circuits_Race_Driver_Constructer;

create table f_presentation.Circuits_Race_Driver_Constructer(
  race_year int,
  race_name string,
  race_date timestamp, 
  circuits_location string,
  driver_name string,
  driver_number int, 
  driver_nationality string,
  team string, 
  grid int, 
  fastest_Lap int, 
  race_time timestamp, 
  points int, 
  position int
)
using parquet
options(path "abfss://udayadbcontainerv1@udayadbsav1.dfs.core.windows.net/Project-2/Presentation-V1/Race_Results_R");

select * from f_presentation.Circuits_Race_Driver_Constructer;


In [0]:
--All-time performance
--Which drivers have competed in at least 50 races
--and how do they rank by their average points per race? 
--Please also report each driver’s total races and total points.
select distinct driver_name, 
                round(avg(points),2) as average_points, 
                count(distinct race_name) as total_races, 
                sum(points) as total_points,
                rank() over(order By round(avg(points),2) desc) as rank
from f_presentation.Circuits_Race_Driver_Constructer
group by driver_name;
--having count(distinct race_name)>= 50;

In [0]:
select * from f_presentation.Circuits_Race_Driver_Constructer;

In [0]:
--Performance in 2011–2020
--“Between the 2011 and 2020 seasons, which drivers started in at least 50 races, 
--how do they rank by average points per race? 
--Include for each driver their total races and total points scored over that period.”
select driver_name,
       round(avg(points),2) as average_points,
       rank() over (order by round(avg(points),2)desc) as rank,
       count(distinct race_name) as total_races,
       sum(distinct points) as total_points
from f_presentation.Circuits_Race_Driver_Constructer
where race_year between 2011 and 2020
group by driver_name;
--having count(distinct race_name)>=50

In [0]:
--Performance in 2001–2010
--“Between the 2001 and 2010 seasons, which drivers started in at least 50 races, 
--and how do they rank by average points per race? 
--Include for each driver their total races and total points scored over that period.”
select driver_name,
       round(avg(points),2) as average_points,
       rank() over (order by round(avg(points),2)desc) as rank,
       count(1) as total_races,
       sum(distinct points) as total_points
from f_presentation.Circuits_Race_Driver_Constructer
where race_year between 2001 and 2010
group by driver_name
having count(1)>=50;

In [0]:
-- Databricks notebook source
SELECT driver_name,
       COUNT(1) AS total_races,
       SUM(points) AS total_points,
       AVG(points) AS avg_points
from f_presentation.Circuits_Race_Driver_Constructer
GROUP BY driver_name
HAVING COUNT(1) >= 50
ORDER BY avg_points DESC;

In [0]:
-- COMMAND ----------

SELECT driver_name,
       COUNT(1) AS total_races,
       SUM(points) AS total_points,
       AVG(points) AS avg_points
from f_presentation.Circuits_Race_Driver_Constructer
 WHERE race_year BETWEEN 2011 AND 2020
GROUP BY driver_name
HAVING COUNT(1) >= 50
ORDER BY avg_points DESC;

In [0]:
-- COMMAND ----------

SELECT driver_name,
       COUNT(1) AS total_races,
       SUM(points) AS total_points,
       AVG(points) AS avg_points
from f_presentation.Circuits_Race_Driver_Constructer
 WHERE race_year BETWEEN 2001 AND 2010
GROUP BY driver_name
HAVING COUNT(1) >= 50
ORDER BY avg_points DESC;

In [0]:
--Here are the business/analysis questions that each of those SQL queries is designed to answer:
--Overall Performance Across All Years
--Which teams have competed in at least 100 races in our entire results table, 
--and what are their total points scored 
--average points per --race
--ranked from highest to lowest average?
select team, count(1) as total_races, sum(points) as total_points, round(avg(points),2) as average_points,
rank()over(order by round(avg(points),2)desc) as rank
from f_presentation.Circuits_Race_Driver_Constructer
group by team
having count(1)>=100
order by average_points desc;

In [0]:
--Performance in the 2011–2020 Decade
--“Restricting to races held between 2011 and 2020, 
--which teams meet the 100-race minimum, and of those, 
--which have the highest total and average points per race—ordered by average points descending?”
select 
      team,
      count(1) as total_races, 
      round(avg(points),2) as avg_points,
      sum(points) as total_points
from f_presentation.Circuits_Race_Driver_Constructer
where race_year between 2011 and 2020
group by team
having count(1)>=100
order by avg_points desc;


In [0]:
--Performance in the 2001-2011 Period
--“For races from 2001 through 2011, 
--which teams have at least 100 starts, 
--and what are their aggregate and per-race point averages—sorted by --average points from highest to lowest?”

select
        team,
        count(1) as total_races,
        round(avg(points),2) as average_points,
        sum(points) as total_points
        from f_presentation.Circuits_Race_Driver_Constructer
        where race_year between 2001 and 2011
        group by team
        having count(1)>=100
        order by average_points desc;

In [0]:

create or replace temp view dominant_Drivers
as
select * from f_presentation.Circuits_Race_Driver_Constructer;


In [0]:
--Here are the business-oriented questions each of those SQL blocks is asking:
--Identifying the Dominant Drivers
--“Which drivers have started at least 50 races in our entire results set, 
--and what are their total races, total points scored, 
--average points per race, and overall rank by average points?”
create or replace temp view dominant_Drivers_1
as
select 
      driver_name,
      count(1) as total_races,
      sum(points) as total_points,
      round(avg(points),2) as average_points,
      rank()over(order by round(avg(points),2)desc) as rank
from f_presentation.Circuits_Race_Driver_Constructer
group by driver_name
having count(1)>=50;

In [0]:
--Year-by-Year Performance of the Top 10 Drivers
--“For the drivers ranked in the top 10 by career average points (from the view above), 
--what were their total races, total points, and average --points in each individual season—sorted by season 
--and then by average points descending?”

create or replace temp view year_by_year
as
select
        driver_name,
        race_year,
        count(1) as total_races,
        sum(points) as total_points,
        round(avg(points),2) as average_points
from f_presentation.Circuits_Race_Driver_Constructer
group by driver_name, race_year
order by average_points desc limit 10;

select * from year_by_year;

In [0]:
--(Repeated) Year-by-Year Performance
--The next two blocks are identical to #2, 
--so they’re essentially restating the same question:
--“How did each of those top-10 drivers perform in terms of races, 
--total points, and average points in every year, ordered by year and highest average?”

create or replace temp view repeated_year_by_year
as
select
        driver_name,
        race_year,
        count(1) as total_races,
        sum(points) as total_points,
        round(avg(points),2) as average_points
from f_presentation.Circuits_Race_Driver_Constructer
group by driver_name, race_year
order by average_points desc limit 10;

select * from repeated_year_by_year;